In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP01 - TP Jurisdictions
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   What number of third party relationships with third parties are in jurisdictions 
   considered a high risk for bribery and corruption during the assessment period?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. COUNTRY ARRAY PARSING: The location_of_tp, country_prod_serv_originates, and 
      Td_Countries_Impacted columns contain comma-separated lists of countries 
      (e.g., 'Canada, India'). Uses array_contains(split(...)) to safely evaluate 
      each country against the High Risk list without triggering false positives.
      Rule: If all three of these columns are blank, it automatically defaults to High Risk.
   3. COMMA EXCEPTION HANDLING: Protects specific unit names containing commas using 
      the '[COMMA]' placeholder swap to prevent improper splitting during explosion.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      LOB lists (owning_lob and lob_using) into individual rows. Restores '[COMMA]' 
      to a real ',' after the split.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax.
   6. DUAL-BRIDGE JOIN: Matches on either the String Name OR the Numeric BusinessID 
      provided in the mapping table's Assessable_Unit_ID column.
   7. DEDUPLICATION: Uses COUNT(DISTINCT EngagementNumber) per AU.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings (Accommodates mixed ID/Name format)
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

High_Risk_Countries AS (
    -- 3. Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    -- 4. Extract data, evaluate Missing Jurisdiction, and apply Comma Protection
    SELECT 
        EngagementNumber,
        
        -- EXCEPTION DICTIONARY: Chain replaces to shield internal commas from split()
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using,
        
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        
        -- Rule: Automatically flag as high risk if no countries are provided at all
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

High_Risk_Engagements AS (
    -- 5. ARRAY PARSER: Filter engagements touching a High Risk country
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.safe_owning_lob,
        tp.safe_lob_using
    FROM Third_Party_Risk_Data tp
    
    -- Safely splits the comma-separated country lists and checks for existence in the High Risk table
    LEFT JOIN High_Risk_Countries h1 
        ON array_contains(split(UPPER(REPLACE(tp.location_of_tp, ', ', ',')), ','), h1.CountryName)
    LEFT JOIN High_Risk_Countries h2 
        ON array_contains(split(UPPER(REPLACE(tp.country_prod_serv_originates, ', ', ',')), ','), h2.CountryName)
    LEFT JOIN High_Risk_Countries h3 
        ON array_contains(split(UPPER(REPLACE(tp.Td_Countries_Impacted, ', ', ',')), ','), h3.CountryName)
        
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Flattened_LOBs AS (
    -- 6. FLATTEN RULE: Explode comma-separated LOB strings into individual rows and restore commas
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM High_Risk_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 7. Bridge mappings to Master List and count the matches
    SELECT 
        mast.BusinessID,
        -- DEDUPLICATION: Count distinct engagements to avoid double-counting expanded rows
        COUNT(DISTINCT f.EngagementNumber) AS High_Risk_Count
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
        
    -- DUAL-BRIDGE JOIN: Supports mapping table's mix of Numeric ID or String Name
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
        
    GROUP BY mast.BusinessID
)

-- 8. Final Output: Strictly structured per Master Anchor
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP01' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP01 - High Risk Jurisdictions Traceability
   PURPOSE: Verifies the Country Array Parser, the Exception Dictionary protection, 
   the LOB flattening, and the Dual-Bridge Master AU lookup status.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Filtered_TP_Data AS (
    SELECT 
        p.EngagementNumber,
        p.owning_lob AS Raw_Owning_LOB,
        p.lob_using AS Raw_Lob_Using,
        
        -- Exception Dictionary Protection
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using,
            
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 'Yes - All Columns Blank'
             ELSE 'No' END AS Missing_Jurisdiction_Flag
             
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    
    -- COUNTRY ARRAY PARSER: Evaluates multi-select dropdown fields
    LEFT JOIN High_Risk_Countries h1 
        ON array_contains(split(UPPER(REPLACE(p.location_of_tp, ', ', ',')), ','), h1.CountryName)
    LEFT JOIN High_Risk_Countries h2 
        ON array_contains(split(UPPER(REPLACE(p.country_prod_serv_originates, ', ', ',')), ','), h2.CountryName)
    LEFT JOIN High_Risk_Countries h3 
        ON array_contains(split(UPPER(REPLACE(p.Td_Countries_Impacted, ', ', ',')), ','), h3.CountryName)
        
    WHERE (h1.CountryName IS NOT NULL OR h2.CountryName IS NOT NULL OR h3.CountryName IS NOT NULL)
       OR (COALESCE(TRIM(p.location_of_tp), '') = '' AND COALESCE(TRIM(p.country_prod_serv_originates), '') = '' AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '')
),

Flattened AS (
    SELECT 
        f.EngagementNumber,
        f.location_of_tp,
        f.country_prod_serv_originates,
        f.Td_Countries_Impacted,
        f.Missing_Jurisdiction_Flag,
        f.Raw_Owning_LOB,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Exploded_Chunk
    FROM Filtered_TP_Data f
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

-- Output columns:
-- EngagementNumber: Third-party engagement identifier being traced.
-- Raw_Location_Of_TP: Original location-of-third-party field from source data.
-- Raw_Country_Originates: Original product/service origin country field from source data.
-- Raw_TD_Countries_Impacted: Original impacted-countries field from source data.
-- Missing_Jurisdiction_Flag: Indicates all three jurisdiction fields were blank.
-- Original_Owning_LOB: Original owning LOB text before flattening.
-- Final_Restored_Chunk: Flattened LOB chunk after exception handling restored commas.
-- Matched_In_Mapping_Table: TPRM unit text matched in the mapping table.
-- Mapping_Table_ID_Or_Name: Assessable-unit value pulled from the mapping table for bridging.
-- Bridged_Master_ID: Master AU ID reached after the dual-bridge step.
-- Master_List_Status: Shows whether the mapped record successfully bridged to the master AU list.
SELECT DISTINCT
    f.EngagementNumber,
    f.location_of_tp AS Raw_Location_Of_TP,
    f.country_prod_serv_originates AS Raw_Country_Originates,
    f.Td_Countries_Impacted AS Raw_TD_Countries_Impacted,
    f.Missing_Jurisdiction_Flag,
    f.Raw_Owning_LOB AS Original_Owning_LOB,
    f.Exploded_Chunk AS Final_Restored_Chunk,
    map.TPRM_Units AS Matched_In_Mapping_Table,
    map.Assessable_Unit_ID AS Mapping_Table_ID_Or_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING' ELSE 'SUCCESS' END AS Master_List_Status
    
FROM Flattened f
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Exploded_Chunk) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    
-- DUAL-BRIDGE JOIN
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
    OR TRIM(map.Assessable_Unit_ID) = TRIM(mast.Master_Numeric_ID)
ORDER BY f.EngagementNumber;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE 2: TP01 - AU Level Response Summary
   PURPOSE: One row per AU showing the bridge keys and how the TP01 response count
   was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

High_Risk_Engagements AS (
    SELECT DISTINCT 
        p.EngagementNumber,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(p.lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1 ELSE 0 END AS Missing_Jurisdiction_Flag,
        CASE WHEN h1.CountryName IS NOT NULL OR h2.CountryName IS NOT NULL OR h3.CountryName IS NOT NULL THEN 1 ELSE 0 END AS High_Risk_Country_Flag
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    LEFT JOIN High_Risk_Countries h1 
        ON array_contains(split(UPPER(REPLACE(p.location_of_tp, ', ', ',')), ','), h1.CountryName)
    LEFT JOIN High_Risk_Countries h2 
        ON array_contains(split(UPPER(REPLACE(p.country_prod_serv_originates, ', ', ',')), ','), h2.CountryName)
    LEFT JOIN High_Risk_Countries h3 
        ON array_contains(split(UPPER(REPLACE(p.Td_Countries_Impacted, ', ', ',')), ','), h3.CountryName)
    WHERE (h1.CountryName IS NOT NULL OR h2.CountryName IS NOT NULL OR h3.CountryName IS NOT NULL)
       OR (COALESCE(TRIM(p.location_of_tp), '') = '' AND COALESCE(TRIM(p.country_prod_serv_originates), '') = '' AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '')
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber,
        Missing_Jurisdiction_Flag,
        High_Risk_Country_Flag,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM High_Risk_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Resolved_Bridge AS (
    SELECT 
        mast.BusinessID,
        mast.AU_Name,
        mast.Business_Segment,
        mast.In_ABAC_AU_List,
        f.EngagementNumber,
        f.Missing_Jurisdiction_Flag,
        f.High_Risk_Country_Flag,
        f.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
),

AU_Debug AS (
    SELECT 
        BusinessID,
        MAX(AU_Name) AS AU_Name,
        MAX(Business_Segment) AS Business_Segment,
        MAX(In_ABAC_AU_List) AS In_ABAC_AU_List,
        COUNT(DISTINCT EngagementNumber) AS Counted_Engagement_Count,
        COUNT(DISTINCT CASE WHEN Missing_Jurisdiction_Flag = 1 THEN EngagementNumber END) AS Missing_Jurisdiction_Engagement_Count,
        COUNT(DISTINCT CASE WHEN High_Risk_Country_Flag = 1 THEN EngagementNumber END) AS High_Risk_Country_Engagement_Count,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(TPRM_Units))) AS Matched_TPRM_Units_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Mapping_ID_Or_Name))) AS Mapping_Bridge_Value_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(EngagementNumber))) AS Engagement_List
    FROM Resolved_Bridge
    GROUP BY BusinessID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final distinct-engagement count returned for TP01.
-- Matched_TPRM_Units_List: TPRM unit values that matched this AU during bridging.
-- Mapping_Bridge_Value_List: Mapping-table assessable-unit values that bridged this AU.
-- Engagement_List: Distinct engagement numbers counted for this AU.
-- Counted_Engagement_Count: Final distinct-engagement total.
-- High_Risk_Country_Engagement_Count: Count of counted engagements triggered by high-risk-country logic.
-- Missing_Jurisdiction_Engagement_Count: Count of counted engagements triggered by all-jurisdictions-blank logic.
-- Calculation_Formula: Plain-English explanation of how Response was produced.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'TP01' AS QuestionID,
    COALESCE(CAST(d.Counted_Engagement_Count AS STRING), '0') AS Response,
    COALESCE(d.Matched_TPRM_Units_List, 'n/a') AS Matched_TPRM_Units_List,
    COALESCE(d.Mapping_Bridge_Value_List, 'n/a') AS Mapping_Bridge_Value_List,
    COALESCE(d.Engagement_List, 'n/a') AS Engagement_List,
    COALESCE(d.Counted_Engagement_Count, 0) AS Counted_Engagement_Count,
    COALESCE(d.High_Risk_Country_Engagement_Count, 0) AS High_Risk_Country_Engagement_Count,
    COALESCE(d.Missing_Jurisdiction_Engagement_Count, 0) AS Missing_Jurisdiction_Engagement_Count,
    CASE 
        WHEN COALESCE(d.Counted_Engagement_Count, 0) = 0 THEN '0 because no high-risk or missing-jurisdiction engagement bridged to this AU'
        ELSE CONCAT(
            'Count distinct engagements mapped to this AU after high-risk jurisdiction or missing-jurisdiction logic. Result=',
            CAST(d.Counted_Engagement_Count AS STRING), '; country-match engagements=', CAST(d.High_Risk_Country_Engagement_Count AS STRING),
            '; missing-jurisdiction engagements=', CAST(d.Missing_Jurisdiction_Engagement_Count AS STRING)
        )
    END AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN AU_Debug d
    ON mast.BusinessID = d.BusinessID
ORDER BY mast.BusinessID;